# Build Head-to-Head Records - Silver Layer

Calculates historical head-to-head statistics between all team pairings.

## Target Schema
* **team_a_id** (INT, part of composite PK)
* **team_a_name** (STRING)
* **team_b_id** (INT, part of composite PK)
* **team_b_name** (STRING)
* **total_meetings** (INT)
* **team_a_wins** (INT)
* **draws** (INT)
* **team_b_wins** (INT)
* **team_a_goals** (INT)
* **team_b_goals** (INT)
* **last_5_results** (STRING) - serialized array of recent results from team_a perspective
* **last_updated** (TIMESTAMP)

## Process
1. Read matches from bronze layer
2. Create directional team pairings (A vs B)
3. Aggregate historical statistics per pairing
4. Calculate wins/draws/goals
5. Collect last 5 results
6. Write to Unity Catalog silver table

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import IntegerType, StringType
from config.paths import MATCHES_BRONZE

# Initialize Spark session
spark = SparkSession.builder.appName("BuildH2HRecords").getOrCreate()

print("=" * 80)
print("Building Head-to-Head Records - Silver Layer")
print("=" * 80)

In [0]:
# Read matches from bronze layer
print("\n[1/5] Reading matches from bronze layer...")
df_matches_raw = spark.read.format("delta").load(MATCHES_BRONZE)
print(f"   Total rows loaded: {df_matches_raw.count():,}")

# Deduplicate matches
df_matches = df_matches_raw.dropDuplicates(["match_id"])
print(f"   Unique matches after deduplication: {df_matches.count():,}")

# Show sample
print("\nSample matches:")
display(df_matches.select(
    "match_id", "match_date",
    "home_team.home_team_id", "home_team.home_team_name",
    "away_team.away_team_id", "away_team.away_team_name",
    "home_score", "away_score"
).limit(3))

In [0]:
# Create team pairings (A vs B) from both home and away perspectives
print("\n[2/5] Creating directional team pairings...")

# Home team as team_a, away team as team_b
df_home_perspective = df_matches.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("home_team.home_team_id").alias("team_a_id"),
    F.col("home_team.home_team_name").alias("team_a_name"),
    F.col("away_team.away_team_id").alias("team_b_id"),
    F.col("away_team.away_team_name").alias("team_b_name"),
    F.col("home_score").alias("team_a_score"),
    F.col("away_score").alias("team_b_score"),
    F.when(F.col("home_score") > F.col("away_score"), "A")
     .when(F.col("home_score") == F.col("away_score"), "D")
     .otherwise("B").alias("result")
)

# Away team as team_a, home team as team_b
df_away_perspective = df_matches.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("away_team.away_team_id").alias("team_a_id"),
    F.col("away_team.away_team_name").alias("team_a_name"),
    F.col("home_team.home_team_id").alias("team_b_id"),
    F.col("home_team.home_team_name").alias("team_b_name"),
    F.col("away_score").alias("team_a_score"),
    F.col("home_score").alias("team_b_score"),
    F.when(F.col("away_score") > F.col("home_score"), "A")
     .when(F.col("away_score") == F.col("home_score"), "D")
     .otherwise("B").alias("result")
)

# Union both perspectives
df_pairings = df_home_perspective.union(df_away_perspective)
print(f"   Total directional pairings: {df_pairings.count():,}")

print("\nSample pairings:")
display(df_pairings.limit(3))

In [0]:
# Aggregate statistics for each team_a vs team_b pairing
print("\n[3/5] Aggregating statistics per pairing...")

df_aggregated = df_pairings.groupBy(
    "team_a_id", "team_a_name", "team_b_id", "team_b_name"
).agg(
    F.count("*").alias("total_meetings"),
    F.sum(F.when(F.col("result") == "A", 1).otherwise(0)).alias("team_a_wins"),
    F.sum(F.when(F.col("result") == "D", 1).otherwise(0)).alias("draws"),
    F.sum(F.when(F.col("result") == "B", 1).otherwise(0)).alias("team_b_wins"),
    F.sum("team_a_score").alias("team_a_goals"),
    F.sum("team_b_score").alias("team_b_goals")
)

print(f"   Unique pairings: {df_aggregated.count():,}")

print("\nSample aggregated pairings:")
display(df_aggregated.limit(3))

In [0]:
# Collect last 5 results for each pairing (from team_a perspective)
print("\n[4/5] Collecting last 5 results per pairing...")

# Window to get last 5 matches per pairing, ordered by date descending
window_last5 = Window.partitionBy("team_a_id", "team_b_id").orderBy(F.desc("match_date")).rowsBetween(0, 4)

# Collect results into array
df_with_results = df_pairings.withColumn(
    "result_array",
    F.collect_list("result").over(window_last5)
)

# Get the most recent record for each pairing (which will have the array)
window_latest = Window.partitionBy("team_a_id", "team_b_id").orderBy(F.desc("match_date"))

df_last5_results = df_with_results.withColumn(
    "row_num",
    F.row_number().over(window_latest)
).filter(
    F.col("row_num") == 1
).select(
    "team_a_id",
    "team_b_id",
    F.array_join("result_array", ",").alias("last_5_results")
)

print(f"   Last 5 results collected for {df_last5_results.count():,} pairings")

print("\nSample last 5 results:")
display(df_last5_results.limit(3))

In [0]:
# Join aggregated stats with last 5 results
print("\n[5/5] Creating final dataset...")

df_final = df_aggregated.join(
    df_last5_results,
    on=["team_a_id", "team_b_id"],
    how="left"
).withColumn(
    "last_updated",
    F.current_timestamp()
).select(
    F.col("team_a_id").cast("int"),
    F.col("team_a_name"),
    F.col("team_b_id").cast("int"),
    F.col("team_b_name"),
    F.col("total_meetings").cast("int"),
    F.col("team_a_wins").cast("int"),
    F.col("draws").cast("int"),
    F.col("team_b_wins").cast("int"),
    F.col("team_a_goals").cast("int"),
    F.col("team_b_goals").cast("int"),
    F.col("last_5_results"),
    F.col("last_updated")
)

print(f"   Final dataset: {df_final.count():,} head-to-head records")

print("\nSample final data:")
df_final.show(5, truncate=False)

# Write to Unity Catalog
print("\nWriting to Unity Catalog table: matchpulse.silver.h2h_records...")
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("matchpulse.silver.h2h_records")

print("✓ Successfully written to matchpulse.silver.h2h_records")

In [0]:
%sql
-- Verify the table was created and query sample data
SELECT 
    team_a_name,
    team_b_name,
    total_meetings,
    team_a_wins,
    draws,
    team_b_wins,
    team_a_goals,
    team_b_goals,
    last_5_results
FROM matchpulse.silver.h2h_records
WHERE total_meetings >= 5
ORDER BY total_meetings DESC
LIMIT 20;

In [0]:
# Check what competitions exist in bronze matches
print("Competitions in bronze matches:")
df_matches.groupBy("competition.competition_name", "competition.country_name") \
    .agg(F.count("*").alias("match_count")) \
    .orderBy(F.desc("match_count")) \
    .show(20, truncate=False)

# Check what competitions made it to h2h_records
print("\nChecking h2h_records table for competition diversity...")
df_h2h = spark.read.table("matchpulse.silver.h2h_records")

# Sample some team names to see variety
print(f"\nTotal unique teams in h2h_records:")
teams_a = df_h2h.select("team_a_id", "team_a_name").distinct()
teams_b = df_h2h.select("team_b_id", "team_b_name").withColumnRenamed("team_b_id", "team_a_id").withColumnRenamed("team_b_name", "team_a_name")
all_teams = teams_a.union(teams_b).distinct().orderBy("team_a_name")
print(f"   {all_teams.count()} unique teams")

print("\nSample of all teams:")
all_teams.show(30, truncate=False)

In [0]:
%sql
-- Sample h2h records from different competitions/leagues
WITH premier_league AS (
  SELECT * FROM matchpulse.silver.h2h_records 
  WHERE team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool')
    AND team_b_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 'Tottenham', 'Manchester City')
  ORDER BY total_meetings DESC LIMIT 5
),
serie_a AS (
  SELECT * FROM matchpulse.silver.h2h_records 
  WHERE team_a_name IN ('Juventus', 'Inter Milan', 'AC Milan', 'AS Roma')
    AND team_b_name IN ('Juventus', 'Inter Milan', 'AC Milan', 'AS Roma', 'Napoli')
  ORDER BY total_meetings DESC LIMIT 5
),
ligue_1 AS (
  SELECT * FROM matchpulse.silver.h2h_records 
  WHERE team_a_name IN ('Paris Saint-Germain', 'Lyon', 'Marseille', 'AS Monaco')
    AND team_b_name IN ('Paris Saint-Germain', 'Lyon', 'Marseille', 'AS Monaco', 'Lille')
  ORDER BY total_meetings DESC LIMIT 5
)
SELECT 'Premier League' as league, team_a_name, team_b_name, total_meetings, team_a_wins, draws, team_b_wins
FROM premier_league
UNION ALL
SELECT 'Serie A' as league, team_a_name, team_b_name, total_meetings, team_a_wins, draws, team_b_wins
FROM serie_a
UNION ALL
SELECT 'Ligue 1' as league, team_a_name, team_b_name, total_meetings, team_a_wins, draws, team_b_wins
FROM ligue_1
ORDER BY league, total_meetings DESC;

# Advanced Visualizations

Production-level visual analytics for head-to-head records.

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Calculate win rates for all pairings
df_winrate = spark.sql("""
    SELECT 
        team_a_name,
        team_b_name,
        total_meetings,
        ROUND(100.0 * team_a_wins / total_meetings, 1) as win_rate,
        ROUND(100.0 * draws / total_meetings, 1) as draw_rate,
        ROUND(100.0 * team_b_wins / total_meetings, 1) as loss_rate
    FROM matchpulse.silver.h2h_records
    WHERE total_meetings >= 5
""").toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Win rate distribution
axes[0].hist(df_winrate['win_rate'], bins=20, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].axvline(df_winrate['win_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_winrate['win_rate'].mean():.1f}%")
axes[0].set_xlabel('Win Rate (%)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Win Rate Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Draw rate distribution
axes[1].hist(df_winrate['draw_rate'], bins=20, color='#95a5a6', alpha=0.7, edgecolor='black')
axes[1].axvline(df_winrate['draw_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_winrate['draw_rate'].mean():.1f}%")
axes[1].set_xlabel('Draw Rate (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Draw Rate Distribution', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Loss rate distribution
axes[2].hist(df_winrate['loss_rate'], bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[2].axvline(df_winrate['loss_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_winrate['loss_rate'].mean():.1f}%")
axes[2].set_xlabel('Loss Rate (%)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[2].set_title('Loss Rate Distribution', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Match Outcome Distribution Analysis (Pairings with 5+ Meetings)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nStatistics (n={len(df_winrate)} pairings):")
print(f"Average Win Rate: {df_winrate['win_rate'].mean():.1f}%")
print(f"Average Draw Rate: {df_winrate['draw_rate'].mean():.1f}%")
print(f"Average Loss Rate: {df_winrate['loss_rate'].mean():.1f}%")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Deduplicate bidirectional pairings and identify league
df_league_rivals = spark.sql("""
    WITH league_mapping AS (
        -- Map teams to their leagues based on known team names
        SELECT team_a_id, team_a_name,
            CASE 
                WHEN team_a_name IN ('Barcelona', 'Real Madrid', 'Atlético Madrid', 'Valencia', 'Sevilla', 
                                     'Athletic Club', 'Real Betis', 'Villarreal', 'Real Sociedad', 'Espanyol',
                                     'Getafe', 'Levante UD', 'Granada', 'Racing Santander', 'Real Zaragoza',
                                     'Real Valladolid', 'RC Deportivo La Coruña', 'Almería') THEN 'La Liga'
                WHEN team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 'Manchester City',
                                     'Tottenham', 'West Ham United', 'Southampton', 'Aston Villa', 'Leicester City') THEN 'Premier League'
                WHEN team_a_name IN ('Juventus', 'Inter Milan', 'AC Milan', 'AS Roma', 'Napoli', 'Lazio',
                                     'Atalanta', 'Palermo', 'Sassuolo', 'Bologna', 'Carpi') THEN 'Serie A'
                WHEN team_a_name IN ('Paris Saint-Germain', 'Lyon', 'Marseille', 'AS Monaco', 'Lille',
                                     'Bordeaux', 'Angers', 'Bastia', 'Auxerre', 'AC Ajaccio') THEN 'Ligue 1'
                WHEN team_a_name IN ('Bayern Munich', 'Borussia Dortmund', 'Bayer Leverkusen', 'Borussia Mönchengladbach',
                                     'Augsburg', 'Bochum') THEN 'Bundesliga'
                ELSE 'Other'
            END as league
        FROM matchpulse.silver.h2h_records
        GROUP BY team_a_id, team_a_name
    ),
    deduplicated AS (
        SELECT h.*,
               l1.league as team_a_league,
               l2.league as team_b_league
        FROM matchpulse.silver.h2h_records h
        INNER JOIN league_mapping l1 ON h.team_a_id = l1.team_a_id
        INNER JOIN league_mapping l2 ON h.team_b_id = l2.team_a_id
        WHERE h.team_a_id < h.team_b_id  -- Deduplicate: only show A vs B, not B vs A
          AND l1.league = l2.league      -- Same league matchups only
          AND l1.league != 'Other'
          AND h.total_meetings >= 5
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY team_a_league ORDER BY total_meetings DESC) as rn
        FROM deduplicated
    )
    SELECT 
        team_a_league as league,
        CONCAT(team_a_name, ' vs ', team_b_name) as rivalry,
        total_meetings,
        team_a_wins + team_b_wins as total_decisive,
        draws,
        ROUND(100.0 * draws / total_meetings, 1) as draw_pct
    FROM ranked
    WHERE rn <= 3  -- Top 3 rivalries per league
    ORDER BY league, total_meetings DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(14, 8))

# Create grouped bar chart
leagues = df_league_rivals['league'].unique()
y_positions = {}
current_y = 0

for league in sorted(leagues):
    league_data = df_league_rivals[df_league_rivals['league'] == league]
    for idx, row in league_data.iterrows():
        y_positions[f"{league}_{idx}"] = current_y
        
        # Stacked bars: decisive matches + draws
        ax.barh(current_y, row['total_decisive'], color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.barh(current_y, row['draws'], left=row['total_decisive'], color='#95a5a6', alpha=0.8, edgecolor='black', linewidth=0.5)
        
        current_y += 1
    current_y += 0.5  # Add spacing between leagues

# Customize
y_labels = [f"{row['rivalry']} ({row['league']})" for _, row in df_league_rivals.iterrows()]
y_ticks = list(range(len(df_league_rivals)))

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=10)
ax.set_xlabel('Total Meetings', fontsize=12, fontweight='bold')
ax.set_title('Top Rivalries by League\n(Deduplicated - Top 3 per League)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', alpha=0.8, label='Decisive Matches'),
    Patch(facecolor='#95a5a6', alpha=0.8, label='Draws')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

# Add value labels
for i, (idx, row) in enumerate(df_league_rivals.iterrows()):
    ax.text(row['total_meetings'] + 0.5, i, f"{row['total_meetings']}", 
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nShowing {len(df_league_rivals)} unique rivalries across {len(leagues)} leagues")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Calculate win rate distribution by league
df_league_analysis = spark.sql("""
    WITH league_mapping AS (
        SELECT team_a_id, team_a_name,
            CASE 
                WHEN team_a_name IN ('Barcelona', 'Real Madrid', 'Atlético Madrid', 'Valencia', 'Sevilla', 
                                     'Athletic Club', 'Real Betis', 'Villarreal', 'Real Sociedad', 'Espanyol',
                                     'Getafe', 'Levante UD', 'Granada', 'Racing Santander', 'Real Zaragoza',
                                     'Real Valladolid', 'RC Deportivo La Coruña', 'Almería') THEN 'La Liga'
                WHEN team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 'Manchester City',
                                     'Tottenham', 'West Ham United', 'Southampton', 'Aston Villa', 'Leicester City') THEN 'Premier League'
                WHEN team_a_name IN ('Juventus', 'Inter Milan', 'AC Milan', 'AS Roma', 'Napoli', 'Lazio',
                                     'Atalanta', 'Palermo', 'Sassuolo', 'Bologna', 'Carpi') THEN 'Serie A'
                WHEN team_a_name IN ('Paris Saint-Germain', 'Lyon', 'Marseille', 'AS Monaco', 'Lille',
                                     'Bordeaux', 'Angers', 'Bastia', 'Auxerre', 'AC Ajaccio') THEN 'Ligue 1'
                WHEN team_a_name IN ('Bayern Munich', 'Borussia Dortmund', 'Bayer Leverkusen', 'Borussia Mönchengladbach',
                                     'Augsburg', 'Bochum') THEN 'Bundesliga'
                ELSE 'Other'
            END as league
        FROM matchpulse.silver.h2h_records
        GROUP BY team_a_id, team_a_name
    )
    SELECT 
        l.league,
        h.team_a_name,
        h.team_b_name,
        h.total_meetings,
        ROUND(100.0 * h.team_a_wins / h.total_meetings, 1) as win_rate,
        ROUND(100.0 * h.draws / h.total_meetings, 1) as draw_rate
    FROM matchpulse.silver.h2h_records h
    INNER JOIN league_mapping l ON h.team_a_id = l.team_a_id
    WHERE l.league != 'Other'
      AND h.total_meetings >= 5
      AND h.team_a_id < h.team_b_id  -- Deduplicate
""").toPandas()

df_league_analysis['win_rate'] = pd.to_numeric(df_league_analysis['win_rate'])
df_league_analysis['draw_rate'] = pd.to_numeric(df_league_analysis['draw_rate'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot for win rate distribution by league
league_order = ['La Liga', 'Premier League', 'Serie A', 'Ligue 1', 'Bundesliga']
league_data = [df_league_analysis[df_league_analysis['league'] == lg]['win_rate'].values 
               for lg in league_order if lg in df_league_analysis['league'].values]
league_labels = [lg for lg in league_order if lg in df_league_analysis['league'].values]

bp = axes[0].boxplot(league_data, labels=league_labels, patch_artist=True,
                      boxprops=dict(facecolor='#3498db', alpha=0.7),
                      medianprops=dict(color='red', linewidth=2),
                      whiskerprops=dict(color='black'),
                      capprops=dict(color='black'))

axes[0].set_ylabel('Win Rate (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Win Rate Distribution by League', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Bar chart for average draw rate by league
avg_draw_by_league = df_league_analysis.groupby('league')['draw_rate'].mean().sort_values(ascending=False)
colors_draw = plt.cm.YlOrRd(np.linspace(0.3, 0.8, len(avg_draw_by_league)))

axes[1].bar(range(len(avg_draw_by_league)), avg_draw_by_league.values, 
            color=colors_draw, alpha=0.8, edgecolor='black', linewidth=1)
axes[1].set_xticks(range(len(avg_draw_by_league)))
axes[1].set_xticklabels(avg_draw_by_league.index, rotation=45, ha='right')
axes[1].set_ylabel('Average Draw Rate (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Average Draw Rate by League', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(avg_draw_by_league.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')

plt.suptitle('League Competitiveness Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nLeague Statistics:")
for league in league_labels:
    league_df = df_league_analysis[df_league_analysis['league'] == league]
    print(f"\n{league}:")
    print(f"  Matchups analyzed: {len(league_df)}")
    print(f"  Avg win rate: {league_df['win_rate'].mean():.1f}%")
    print(f"  Avg draw rate: {league_df['draw_rate'].mean():.1f}%")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get goal scoring patterns by league
df_goals_by_league = spark.sql("""
    WITH league_mapping AS (
        SELECT team_a_id, team_a_name,
            CASE 
                WHEN team_a_name IN ('Barcelona', 'Real Madrid', 'Atlético Madrid', 'Valencia', 'Sevilla', 
                                     'Athletic Club', 'Real Betis', 'Villarreal', 'Real Sociedad', 'Espanyol',
                                     'Getafe', 'Levante UD', 'Granada', 'Racing Santander', 'Real Zaragoza',
                                     'Real Valladolid', 'RC Deportivo La Coruña', 'Almería') THEN 'La Liga'
                WHEN team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 'Manchester City',
                                     'Tottenham', 'West Ham United', 'Southampton', 'Aston Villa', 'Leicester City') THEN 'Premier League'
                WHEN team_a_name IN ('Juventus', 'Inter Milan', 'AC Milan', 'AS Roma', 'Napoli', 'Lazio',
                                     'Atalanta', 'Palermo', 'Sassuolo', 'Bologna', 'Carpi') THEN 'Serie A'
                WHEN team_a_name IN ('Paris Saint-Germain', 'Lyon', 'Marseille', 'AS Monaco', 'Lille',
                                     'Bordeaux', 'Angers', 'Bastia', 'Auxerre', 'AC Ajaccio') THEN 'Ligue 1'
                WHEN team_a_name IN ('Bayern Munich', 'Borussia Dortmund', 'Bayer Leverkusen', 'Borussia Mönchengladbach',
                                     'Augsburg', 'Bochum') THEN 'Bundesliga'
                ELSE 'Other'
            END as league
        FROM matchpulse.silver.h2h_records
        GROUP BY team_a_id, team_a_name
    )
    SELECT 
        l.league,
        h.team_a_name,
        h.total_meetings,
        ROUND((h.team_a_goals + h.team_b_goals) * 1.0 / h.total_meetings, 2) as avg_total_goals,
        ROUND(h.team_a_goals * 1.0 / h.total_meetings, 2) as avg_goals_scored,
        ROUND(h.team_b_goals * 1.0 / h.total_meetings, 2) as avg_goals_conceded
    FROM matchpulse.silver.h2h_records h
    INNER JOIN league_mapping l ON h.team_a_id = l.team_a_id
    WHERE l.league != 'Other'
      AND h.total_meetings >= 5
      AND h.team_a_id < h.team_b_id
""").toPandas()

df_goals_by_league['avg_total_goals'] = pd.to_numeric(df_goals_by_league['avg_total_goals'])
df_goals_by_league['avg_goals_scored'] = pd.to_numeric(df_goals_by_league['avg_goals_scored'])
df_goals_by_league['avg_goals_conceded'] = pd.to_numeric(df_goals_by_league['avg_goals_conceded'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Violin plot for total goals per match distribution
league_order = ['La Liga', 'Premier League', 'Serie A', 'Ligue 1', 'Bundesliga']
league_labels = [lg for lg in league_order if lg in df_goals_by_league['league'].values]

violin_data = [df_goals_by_league[df_goals_by_league['league'] == lg]['avg_total_goals'].values 
               for lg in league_labels]

parts = axes[0].violinplot(violin_data, positions=range(len(league_labels)), 
                            showmeans=True, showmedians=True)

for pc in parts['bodies']:
    pc.set_facecolor('#3498db')
    pc.set_alpha(0.7)

axes[0].set_xticks(range(len(league_labels)))
axes[0].set_xticklabels(league_labels, rotation=45, ha='right')
axes[0].set_ylabel('Average Total Goals per Match', fontsize=11, fontweight='bold')
axes[0].set_title('Goal Scoring Distribution by League', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Grouped bar chart: Attack vs Defense by league
league_stats = df_goals_by_league.groupby('league').agg({
    'avg_goals_scored': 'mean',
    'avg_goals_conceded': 'mean'
}).reindex(league_labels)

x = np.arange(len(league_labels))
width = 0.35

axes[1].bar(x - width/2, league_stats['avg_goals_scored'], width, 
            label='Avg Goals Scored', color='#2ecc71', alpha=0.8, edgecolor='black')
axes[1].bar(x + width/2, league_stats['avg_goals_conceded'], width, 
            label='Avg Goals Conceded', color='#e74c3c', alpha=0.8, edgecolor='black')

axes[1].set_xticks(x)
axes[1].set_xticklabels(league_labels, rotation=45, ha='right')
axes[1].set_ylabel('Average Goals per Match', fontsize=11, fontweight='bold')
axes[1].set_title('Attack vs Defense by League', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Cross-League Goal Scoring Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nGoal Scoring Statistics by League:")
for league in league_labels:
    league_df = df_goals_by_league[df_goals_by_league['league'] == league]
    print(f"\n{league}:")
    print(f"  Avg total goals per match: {league_df['avg_total_goals'].mean():.2f}")
    print(f"  Avg goals scored: {league_df['avg_goals_scored'].mean():.2f}")
    print(f"  Avg goals conceded: {league_df['avg_goals_conceded'].mean():.2f}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Calculate team-level dominance (aggregate across all opponents)
df_dominance = spark.sql("""
    SELECT 
        team_a_name as team,
        SUM(total_meetings) as total_matches,
        SUM(team_a_wins) as total_wins,
        SUM(draws) as total_draws,
        SUM(team_b_wins) as total_losses,
        ROUND(100.0 * SUM(team_a_wins) / SUM(total_meetings), 1) as win_pct,
        ROUND(100.0 * SUM(draws) / SUM(total_meetings), 1) as draw_pct
    FROM matchpulse.silver.h2h_records
    GROUP BY team_a_name
    HAVING SUM(total_meetings) >= 20
    ORDER BY win_pct DESC
    LIMIT 15
""").toPandas()

# Convert to numeric
df_dominance['win_pct'] = pd.to_numeric(df_dominance['win_pct'])

fig, ax = plt.subplots(figsize=(14, 8))

# Create horizontal bar chart
y_pos = range(len(df_dominance))
colors = plt.cm.RdYlGn(df_dominance['win_pct'] / 100)

bars = ax.barh(y_pos, df_dominance['win_pct'], color=colors, alpha=0.8, edgecolor='black', linewidth=1)

# Customize
ax.set_yticks(y_pos)
ax.set_yticklabels(df_dominance['team'], fontsize=11)
ax.set_xlabel('Win Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Most Dominant Teams\n(Minimum 20 matches across all opponents)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(df_dominance.iterrows()):
    ax.text(row['win_pct'] + 1, i, f"{row['win_pct']:.1f}% ({row['total_wins']}/{row['total_matches']})", 
            va='center', fontsize=9, fontweight='bold')

# Add 50% reference line
ax.axvline(50, color='red', linestyle='--', alpha=0.5, linewidth=2, label='50% (Even)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get Premier League rivalries (deduplicated)
df_pl_rivals = spark.sql("""
    WITH pl_teams AS (
        SELECT DISTINCT team_a_id, team_a_name
        FROM matchpulse.silver.h2h_records
        WHERE team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 
                              'Manchester City', 'Tottenham', 'West Ham United', 
                              'Southampton', 'Aston Villa', 'Leicester City',
                              'Everton', 'Newcastle United', 'Sunderland')
    ),
    pl_matchups AS (
        SELECT h.*
        FROM matchpulse.silver.h2h_records h
        INNER JOIN pl_teams t1 ON h.team_a_id = t1.team_a_id
        INNER JOIN pl_teams t2 ON h.team_b_id = t2.team_a_id
        WHERE h.team_a_id < h.team_b_id  -- Deduplicate
          AND h.total_meetings >= 2
    )
    SELECT 
        CONCAT(team_a_name, ' vs ', team_b_name) as rivalry,
        total_meetings,
        team_a_wins,
        draws,
        team_b_wins,
        team_a_goals,
        team_b_goals,
        ROUND(100.0 * draws / total_meetings, 1) as draw_pct
    FROM pl_matchups
    ORDER BY total_meetings DESC
    LIMIT 12
""").toPandas()

# Convert to numeric
df_pl_rivals['draw_pct'] = pd.to_numeric(df_pl_rivals['draw_pct'])

fig, ax = plt.subplots(figsize=(14, 10))

# Create stacked horizontal bar chart
y_pos = range(len(df_pl_rivals))

ax.barh(y_pos, df_pl_rivals['team_a_wins'], label='Team A Wins', color='#9b59b6', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.barh(y_pos, df_pl_rivals['draws'], left=df_pl_rivals['team_a_wins'], 
        label='Draws', color='#95a5a6', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.barh(y_pos, df_pl_rivals['team_b_wins'], 
        left=df_pl_rivals['team_a_wins'] + df_pl_rivals['draws'], 
        label='Team B Wins', color='#e67e22', alpha=0.85, edgecolor='black', linewidth=0.5)

# Customize
ax.set_yticks(y_pos)
ax.set_yticklabels(df_pl_rivals['rivalry'], fontsize=10)
ax.set_xlabel('Total Meetings', fontsize=12, fontweight='bold')
ax.set_title('Premier League Top Rivalries\nMatch Outcome Distribution', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)

# Add total labels
for i, (idx, row) in enumerate(df_pl_rivals.iterrows()):
    total = row['total_meetings']
    ax.text(total + 0.2, i, f"{total}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nPremier League Statistics:")
print(f"Total rivalries shown: {len(df_pl_rivals)}")
print(f"Average meetings per rivalry: {df_pl_rivals['total_meetings'].mean():.1f}")
print(f"Average draw rate: {df_pl_rivals['draw_pct'].mean():.1f}%")
if len(df_pl_rivals) > 0:
    max_idx = df_pl_rivals['draw_pct'].idxmax()
    print(f"\nMost competitive (highest draw rate): {df_pl_rivals.loc[max_idx, 'rivalry']} ({df_pl_rivals.loc[max_idx, 'draw_pct']:.1f}%)")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get Premier League goal scoring data
df_pl_goals = spark.sql("""
    WITH pl_teams AS (
        SELECT DISTINCT team_a_id, team_a_name
        FROM matchpulse.silver.h2h_records
        WHERE team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 
                              'Manchester City', 'Tottenham', 'West Ham United', 
                              'Southampton', 'Aston Villa', 'Leicester City',
                              'Everton', 'Newcastle United', 'Sunderland')
    )
    SELECT 
        h.team_a_name,
        h.team_b_name,
        h.total_meetings,
        ROUND(h.team_a_goals * 1.0 / h.total_meetings, 2) as avg_goals_scored,
        ROUND(h.team_b_goals * 1.0 / h.total_meetings, 2) as avg_goals_conceded,
        ROUND((h.team_a_goals + h.team_b_goals) * 1.0 / h.total_meetings, 2) as avg_total_goals
    FROM matchpulse.silver.h2h_records h
    INNER JOIN pl_teams t1 ON h.team_a_id = t1.team_a_id
    INNER JOIN pl_teams t2 ON h.team_b_id = t2.team_a_id
    WHERE h.team_a_id < h.team_b_id
      AND h.total_meetings >= 2
""").toPandas()

df_pl_goals['avg_goals_scored'] = pd.to_numeric(df_pl_goals['avg_goals_scored'])
df_pl_goals['avg_goals_conceded'] = pd.to_numeric(df_pl_goals['avg_goals_conceded'])
df_pl_goals['avg_total_goals'] = pd.to_numeric(df_pl_goals['avg_total_goals'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Scatter plot of goals scored vs conceded
scatter = axes[0].scatter(df_pl_goals['avg_goals_scored'], 
                          df_pl_goals['avg_goals_conceded'],
                          s=df_pl_goals['total_meetings']*40,
                          alpha=0.6,
                          c=df_pl_goals['total_meetings'],
                          cmap='plasma',
                          edgecolors='black',
                          linewidth=0.5)

# Add diagonal line
max_val = max(df_pl_goals['avg_goals_scored'].max(), df_pl_goals['avg_goals_conceded'].max())
axes[0].plot([0, max_val], [0, max_val], 'r--', alpha=0.5, linewidth=2, label='Balanced')

axes[0].set_xlabel('Avg Goals Scored per Match', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Avg Goals Conceded per Match', fontsize=11, fontweight='bold')
axes[0].set_title('Premier League: Attack vs Defense\n(Bubble size = Total Meetings)', 
                  fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=10)

# Add colorbar
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Total Meetings', fontsize=10, fontweight='bold')

# Right: Distribution of total goals per match
axes[1].hist(df_pl_goals['avg_total_goals'], bins=15, color='#9b59b6', alpha=0.7, edgecolor='black', linewidth=1)
axes[1].axvline(df_pl_goals['avg_total_goals'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f"Mean: {df_pl_goals['avg_total_goals'].mean():.2f}")
axes[1].axvline(df_pl_goals['avg_total_goals'].median(), color='green', linestyle='--', 
                linewidth=2, label=f"Median: {df_pl_goals['avg_total_goals'].median():.2f}")

axes[1].set_xlabel('Average Total Goals per Match', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Premier League: Goals per Match Distribution', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Premier League Goal Scoring Patterns', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"\nPremier League Goal Statistics:")
print(f"Average goals per match: {df_pl_goals['avg_total_goals'].mean():.2f}")
print(f"Highest scoring rivalry: {df_pl_goals.loc[df_pl_goals['avg_total_goals'].idxmax(), 'team_a_name']} vs {df_pl_goals.loc[df_pl_goals['avg_total_goals'].idxmax(), 'team_b_name']} ({df_pl_goals['avg_total_goals'].max():.2f} goals/match)")
print(f"Most defensive rivalry: {df_pl_goals.loc[df_pl_goals['avg_total_goals'].idxmin(), 'team_a_name']} vs {df_pl_goals.loc[df_pl_goals['avg_total_goals'].idxmin(), 'team_b_name']} ({df_pl_goals['avg_total_goals'].min():.2f} goals/match)")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get top 6 PL teams based on total wins
df_pl_top_teams = spark.sql("""
    WITH pl_teams AS (
        SELECT DISTINCT team_a_id, team_a_name
        FROM matchpulse.silver.h2h_records
        WHERE team_a_name IN ('Arsenal', 'Manchester United', 'Chelsea', 'Liverpool', 
                              'Manchester City', 'Tottenham', 'West Ham United', 
                              'Southampton', 'Aston Villa', 'Leicester City',
                              'Everton', 'Newcastle United', 'Sunderland')
    ),
    team_performance AS (
        SELECT 
            h.team_a_name,
            SUM(h.team_a_wins) as total_wins
        FROM matchpulse.silver.h2h_records h
        INNER JOIN pl_teams t ON h.team_a_id = t.team_a_id
        GROUP BY h.team_a_name
        ORDER BY total_wins DESC
        LIMIT 6
    )
    SELECT 
        h.team_a_name,
        h.team_b_name,
        h.total_meetings,
        ROUND(100.0 * h.team_a_wins / h.total_meetings, 1) as win_pct,
        h.team_a_goals - h.team_b_goals as goal_diff
    FROM matchpulse.silver.h2h_records h
    WHERE h.team_a_name IN (SELECT team_a_name FROM team_performance)
      AND h.team_b_name IN (SELECT team_a_name FROM team_performance)
      AND h.team_a_name != h.team_b_name
""").toPandas()

df_pl_top_teams['win_pct'] = pd.to_numeric(df_pl_top_teams['win_pct'])
df_pl_top_teams['goal_diff'] = pd.to_numeric(df_pl_top_teams['goal_diff'])

# Create pivot tables
pivot_winpct = df_pl_top_teams.pivot_table(
    values='win_pct',
    index='team_a_name',
    columns='team_b_name',
    fill_value=0
)

pivot_goaldiff = df_pl_top_teams.pivot_table(
    values='goal_diff',
    index='team_a_name',
    columns='team_b_name',
    fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Win percentage heatmap
sns.heatmap(pivot_winpct, 
            annot=True, 
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            cbar_kws={'label': 'Win %'},
            linewidths=0.5,
            linecolor='gray',
            ax=axes[0],
            vmin=0, vmax=100)

axes[0].set_title('Premier League Top 6: Win Percentage Matrix\n(Row team vs Column team)', 
                  fontsize=12, fontweight='bold', pad=15)
axes[0].set_xlabel('Opponent', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Team', fontsize=11, fontweight='bold')

# Right: Goal difference heatmap
sns.heatmap(pivot_goaldiff, 
            annot=True, 
            fmt='.0f',
            cmap='RdYlGn',
            center=0,
            cbar_kws={'label': 'Goal Difference'},
            linewidths=0.5,
            linecolor='gray',
            ax=axes[1],
            vmin=-15, vmax=15)

axes[1].set_title('Premier League Top 6: Goal Difference Matrix\n(Row team vs Column team)', 
                  fontsize=12, fontweight='bold', pad=15)
axes[1].set_xlabel('Opponent', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Team', fontsize=11, fontweight='bold')

plt.suptitle('Premier League Top Teams Head-to-Head Performance', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Left - Win %: Green = dominates opponent, Red = struggles against opponent")
print("  Right - Goal Diff: Green = outscores opponent, Red = concedes more to opponent")